# Module 16: Graph Engineering

**Goal:** Express agent control flow as an inspectable graph, and use knowledge graphs to answer the questions vector search can't.

**What you'll learn:**
- State, nodes, edges, and superstep semantics
- Reducers — and the lost-update bug you get without them
- Conditional edges as testable stop conditions; bounded cycles
- Checkpointing: crash resume, human interrupts, time-travel debugging
- Knowledge graph construction: extraction, entity resolution, evidence
- Graph RAG retrieval modes: local, global, path, hybrid
- Temporal edges — facts that expire instead of silently going stale

Everything here runs on the **standard library** — no API key, no graph database. Swap in a real LLM and store afterwards.

## Part 1 — Graphs as Control Flow

### 0. Setup

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Callable
from collections import defaultdict
import re

END, START = "__end__", "__start__"

# Reducers decide how two updates to the SAME field combine.
def last_write_wins(old, new): return new
def append(old, new): return list(old) + list(new)
def add_int(old, new): return old + new

print("✅ Setup complete")

---
### 1. A state graph in one cell

Three pieces only: **state** (typed object every node reads/writes), **nodes** (`state -> partial update`), **edges** (static or conditional).

Execution runs in **supersteps**: everything on the frontier sees the *same* input state, all updates are collected, then merged through the reducers. That is exactly why reducers matter.

In [ ]:
@dataclass
class Checkpoint:
    step: int
    frontier: list
    state: dict


@dataclass
class StateGraph:
    reducers: dict = field(default_factory=dict)
    nodes: dict = field(default_factory=dict)
    edges: dict = field(default_factory=dict)
    branches: dict = field(default_factory=dict)
    checkpoints: list = field(default_factory=list)
    max_steps: int = 50

    def add_node(self, name, fn):
        self.nodes[name] = fn; return self

    def add_edge(self, src, dst):
        self.edges.setdefault(src, []).append(dst); return self

    def add_conditional_edges(self, src, router):
        self.branches[src] = router; return self

    def _merge(self, state, update):
        merged = dict(state)
        for k, v in (update or {}).items():
            reducer = self.reducers.get(k, last_write_wins)
            merged[k] = reducer(merged[k], v) if k in merged else v
        return merged

    def _next(self, node, state):
        if node in self.branches:
            r = self.branches[node](state)
            return [r] if isinstance(r, str) else list(r)
        return list(self.edges.get(node, []))

    def invoke(self, initial, verbose=True):
        state, step = dict(initial), 0
        frontier = self._next(START, state)
        while frontier and END not in frontier:
            if step >= self.max_steps:
                raise RuntimeError("cycle is missing its mechanical stop condition")
            step += 1
            updates = [(n, self.nodes[n](state)) for n in frontier]   # same input state
            if verbose:
                print(f"  superstep {step}: {', '.join(frontier)}")
            for _, u in updates:
                state = self._merge(state, u)                          # then merge
            self.checkpoints.append(Checkpoint(step, list(frontier), dict(state)))
            nxt = []
            for n, _ in updates:
                for t in self._next(n, state):
                    if t not in nxt: nxt.append(t)
            if not nxt: break
            frontier = [n for n in nxt if n != END]
        return state

print("✅ Engine ready")

---
### 2. Reducers are not optional — the lost-update bug

Two parallel branches both return `{"docs": [...]}`. Without a reducer the second silently overwrites the first, and nothing errors.

In [ ]:
def build_parallel(with_reducer: bool):
    g = StateGraph(reducers={"docs": append} if with_reducer else {})
    g.add_node("search_web", lambda s: {"docs": ["web:a", "web:b"]})
    g.add_node("search_kb",  lambda s: {"docs": ["kb:x"]})
    g.add_node("synthesize", lambda s: {"answer": f"{len(s['docs'])} docs used"})
    g.add_conditional_edges(START, lambda s: ["search_web", "search_kb"])   # fan out
    g.add_edge("search_web", "synthesize")                                  # fan in
    g.add_edge("search_kb", "synthesize")
    g.add_edge("synthesize", END)
    return g

for flag in (False, True):
    out = build_parallel(flag).invoke({"docs": [], "answer": ""}, verbose=False)
    print(f"  reducer={str(flag):<5} docs={out['docs']} -> {out['answer']}")

print("\nRule: one writer per scalar field per superstep, or add a reducer.")

---
### 3. Conditional edges + a bounded cycle

A DAG cannot express "retrieve, judge, and if the evidence is weak, rewrite and try again". A graph with a conditional edge can.

Note where the stop conditions live: in `route()`, as ordinary Python you can unit-test without an API key. **Two independent stops** — one semantic (`relevant`), one mechanical (`attempts >= 3`).

In [ ]:
CORPUS = {
    "churn":     "Q4 churn was 3.1%, down from 4.0% in Q3.",
    "revenue":   "Q4 revenue was $210M, up 12% YoY.",
    "headcount": "Headcount grew to 1,240 at end of Q4.",
}

def build_corrective():
    g = StateGraph(reducers={"docs": append, "attempts": add_int})
    g.add_node("retrieve", lambda s: {
        "docs": [t for k, t in CORPUS.items() if k in s["question"].lower()],
        "attempts": 1})
    g.add_node("grade",    lambda s: {"relevant": bool(s["docs"])})
    g.add_node("rewrite",  lambda s: {"question": s["question"] + " churn"})
    g.add_node("generate", lambda s: {"answer": " ".join(s["docs"]) or "No evidence."})

    def route(state):
        if state["relevant"]:        return "generate"   # semantic stop
        if state["attempts"] >= 3:   return "generate"   # mechanical stop
        return "rewrite"

    g.add_edge(START, "retrieve")
    g.add_edge("retrieve", "grade")
    g.add_conditional_edges("grade", route)
    g.add_edge("rewrite", "retrieve")     # <- the cycle
    g.add_edge("generate", END)
    return g

INIT = {"question": "", "docs": [], "attempts": 0, "relevant": False, "answer": ""}

g = build_corrective()
out = g.invoke({**INIT, "question": "what happened to revenue?"})
print(f"  answer: {out['answer']}  (attempts={out['attempts']})\n")

g2 = build_corrective()
out2 = g2.invoke({**INIT, "question": "what is our NPS?"})
print(f"  answer: {out2['answer']}  (attempts={out2['attempts']})")

**Read that second trace carefully.** The cycle terminated correctly and still answered the wrong question: the naive rewrite appended a keyword, pulled an unrelated doc, and the grader passed it.

The graph did its job. The *node* was wrong. That is the argument for node-level evals (Module 09) — graph correctness and node correctness are separate properties.

---
### 4. Checkpoints → resume, interrupts, time travel

One mechanism (persist state after every superstep) buys four capabilities. Below: load an earlier checkpoint, edit state, and re-run from there — the debugger for agent systems.

In LangGraph this is `compile(checkpointer=SqliteSaver(...), interrupt_before=["apply_changes"])` plus a `thread_id`. The pause is durable, so a human approval gate can last days.

In [ ]:
for cp in g2.checkpoints:
    shown = {k: v for k, v in cp.state.items() if k != "docs"}
    print(f"  step {cp.step} after {','.join(cp.frontier):<10} {shown}")

# Time travel: replay from the checkpoint after step 1 with corrected state.
cp = g2.checkpoints[0]
patched = {**cp.state, "docs": ["NPS was 46 in Q4."], "relevant": True}

replay = build_corrective()
replay.branches[START] = lambda s: ["grade"]        # resume at the next node
print("\n  ↩ replaying from step 1 with edited state:")
print("  replayed answer:", replay.invoke(patched)["answer"])

---
## Part 2 — Knowledge Graphs & Graph RAG

Vector RAG retrieves *chunks that look like the question*. Some questions aren't shaped like that:

| Question | Vector RAG | Why |
|---|---|---|
| "What did the Q3 report say about churn?" | ✅ | Answer lives in one chunk |
| "Which suppliers are also our competitors' customers?" | ❌ | Needs a join across documents |
| "How did Acme's ownership change 2019→2024?" | ❌ | Needs traversal **and** time |
| "What are the themes across 4,000 tickets?" | ❌ | Needs aggregation, not top-k |

### 5. Build the graph: closed vocabulary + mandatory evidence

In [ ]:
RELATION_TYPES = {"WORKS_AT", "ACQUIRED", "SUPPLIES", "COMPETES_WITH",
                  "PARTNERED_WITH", "LOCATED_IN"}

@dataclass(frozen=True)
class Edge:
    source: str; type: str; target: str
    evidence: str                      # the sentence it came from — never optional
    valid_from: str = None
    valid_to: str = None               # None -> still believed true

    def active_at(self, as_of):
        if as_of is None: return self.valid_to is None
        if self.valid_from and as_of < self.valid_from: return False
        if self.valid_to and as_of >= self.valid_to:    return False
        return True

    def render(self):
        w = f"  [{self.valid_from or '?'} → {self.valid_to or 'now'}]" if (self.valid_from or self.valid_to) else ""
        return f"{self.source} —[{self.type}]→ {self.target}{w}"


class KnowledgeGraph:
    def __init__(self):
        self.entities, self.aliases, self.edges, self.merge_log = {}, {}, [], []

    @staticmethod
    def _norm(name):
        n = re.sub(r"\b(inc|corp|corporation|co|ltd|llc)\b\.?", "", name.lower().strip())
        return re.sub(r"[^a-z0-9 ]", "", n).strip()

    def resolve(self, name):
        """Cascade cheapest-first: exact -> alias -> normalized. Log every merge."""
        if name in self.entities: return name
        if name in self.aliases:  return self.aliases[name]
        for canon in self.entities:
            if self._norm(canon) == self._norm(name):
                self.aliases[name] = canon
                self.merge_log.append(f"{name!r} -> {canon!r} (normalized)")
                return canon
        return None

    def add_entity(self, name, etype):
        return self.resolve(name) or (self.entities.setdefault(name, etype), name)[1]

    def add_edge(self, source, rtype, target, evidence, valid_from=None):
        if rtype not in RELATION_TYPES:
            raise ValueError(f"relation {rtype!r} not in the closed vocabulary")
        if not evidence:
            raise ValueError("refusing an edge with no evidence — it would be unauditable")
        src = self.resolve(source) or source
        tgt = self.resolve(target) or target
        if valid_from:      # a new fact INVALIDATES the old one; it does not overwrite it
            for i, e in enumerate(self.edges):
                if (e.source, e.type) == (src, rtype) and e.target != tgt and e.valid_to is None:
                    self.edges[i] = Edge(e.source, e.type, e.target, e.evidence,
                                         e.valid_from, valid_from)
        self.edges.append(Edge(src, rtype, tgt, evidence, valid_from))

    def neighbors(self, name, as_of=None):
        c = self.resolve(name)
        return [e for e in self.edges if c in (e.source, e.target) and e.active_at(as_of)]

    def expand(self, anchors, hops=2, as_of=None):
        """LOCAL search — the subgraph around entities named in the question."""
        nodes, out = {self.resolve(a) or a for a in anchors}, []
        for _ in range(hops):
            frontier = set()
            for n in nodes:
                for e in self.neighbors(n, as_of):
                    if e not in out: out.append(e)
                    frontier |= {e.source, e.target}
            if frontier <= nodes: break
            nodes |= frontier
        return out

    def paths(self, start, end, max_hops=4, as_of=None):
        """PATH search — 'how is X connected to Y?'"""
        s, t = self.resolve(start) or start, self.resolve(end) or end
        queue, found = [(s, [])], []
        while queue:
            node, trail = queue.pop(0)
            if len(trail) >= max_hops: continue
            for e in self.neighbors(node, as_of):
                if e in trail: continue
                nxt = e.target if e.source == node else e.source
                (found if nxt == t else queue).append(trail + [e] if nxt == t else (nxt, trail + [e]))
        return found

    def communities(self):
        parent = {n: n for n in self.entities}
        def find(x):
            while parent.setdefault(x, x) != x: x = parent[x]
            return x
        for e in self.edges:
            a, b = find(e.source), find(e.target)
            if a != b: parent[a] = b
        groups = defaultdict(set)
        for n in list(parent): groups[find(n)].add(n)
        return dict(enumerate(groups.values()))

print("✅ KnowledgeGraph ready")

### 6. Extract → resolve → store

`scripted_extract` stands in for an LLM call with a **strict schema** and a closed enum of relation types. Open-ended extraction produces a graph where every document invents new edge names — unqueryable.

Entity resolution is the step that decides whether the graph is useful at all: if "Acme Corp" and "ACME Corporation" stay separate nodes, every traversal dead-ends.

In [ ]:
DOCS = [
    ("Alice Chen joined Acme Corp as VP Engineering in February 2023.",
     [("Alice Chen","PERSON"),("Acme Corp","COMPANY")],
     [("Alice Chen","WORKS_AT","Acme Corp","2023-02-01")]),
    ("ACME Corporation acquired Bolt Systems in 2024 to expand in Berlin.",
     [("ACME Corporation","COMPANY"),("Bolt Systems","COMPANY"),("Berlin","LOCATION")],
     [("ACME Corporation","ACQUIRED","Bolt Systems","2024-05-01"),
      ("Bolt Systems","LOCATED_IN","Berlin",None)]),
    ("Bolt Systems supplies sensor modules to Cirrus Robotics.",
     [("Cirrus Robotics","COMPANY")],
     [("Bolt Systems","SUPPLIES","Cirrus Robotics",None)]),
    ("Cirrus Robotics competes with Northwind Automation in warehouse robotics.",
     [("Northwind Automation","COMPANY")],
     [("Cirrus Robotics","COMPETES_WITH","Northwind Automation",None)]),
    ("Alice Chen left Acme and started at Northwind Automation in July 2025.",
     [], [("Alice Chen","WORKS_AT","Northwind Automation","2025-07-01")]),
    ("Vela Pharma partnered with Kestrel Labs on trial logistics.",
     [("Vela Pharma","COMPANY"),("Kestrel Labs","COMPANY")],
     [("Vela Pharma","PARTNERED_WITH","Kestrel Labs",None)]),
]

kg = KnowledgeGraph()
for text, ents, rels in DOCS:
    for name, etype in ents:
        kg.add_entity(name, etype)
    for s, r, o, vf in rels:
        kg.add_edge(s, r, o, evidence=text, valid_from=vf)

print(f"entities: {len(kg.entities)}   edges: {len(kg.edges)}")
for line in kg.merge_log:
    print("  merged", line)

### 7. Retrieval modes: local, path, global

In [ ]:
print("LOCAL — 'tell me about Bolt Systems' (2 hops):")
for e in kg.expand(["Bolt Systems"], hops=2):
    print("   ", e.render())

print("\nPATH — 'how is Alice Chen connected to Berlin?'")
for as_of in ("2024-06-01", "2024-01-01"):
    trails = kg.paths("Alice Chen", "Berlin", as_of=as_of)
    print(f"  as of {as_of}:")
    if not trails:
        print("    no active path (the acquisition hadn't happened yet)")
    for t in trails:
        print("    " + "  =>  ".join(e.render() for e in t))

print("\nGLOBAL — community summaries (index-time cost, corpus-level answers):")
for cid, members in kg.communities().items():
    inner = [e for e in kg.edges if e.source in members and e.target in members]
    kinds = ", ".join(sorted({e.type for e in inner})) or "no"
    print(f"  community {cid}: {len(members)} entities linked by {kinds} relations")

No single chunk contains both "Alice Chen" and "Berlin" — the answer only exists as a **path**. And the path depends on *when* you ask, which temporal edges make explicit instead of quietly wrong.

**Global search** is what makes Graph RAG distinctive: cluster once at index time, summarize each community once, then answer "what are the main themes?" by mapping over summaries rather than retrieving top-k chunks.

### 8. Temporal facts and guardrails

In [ ]:
print("'Where does Alice Chen work?' — asked as of three dates:")
for as_of in ("2024-01-01", "2025-01-01", "2026-01-01"):
    jobs = [e for e in kg.neighbors("Alice Chen", as_of) if e.type == "WORKS_AT"]
    print(f"  as of {as_of}: {jobs[0].target if jobs else 'unknown'}")
print("The 2023 edge was invalidated, not overwritten — history survives.\n")

for bad in [("Alice Chen","MENTORS","Bob","a sentence"),          # not in vocabulary
            ("Alice Chen","WORKS_AT","Acme Corp","")]:            # no evidence
    try:
        kg.add_edge(*bad)
    except ValueError as exc:
        print("  rejected:", exc)

---
## Hybrid retrieval — the default production shape

Most real traffic is best served by *both*: vector search for passages (what looks like the question) plus graph expansion for structure (what the question is about), re-ranked together.

```python
def hybrid_retrieve(query, k=8):
    passages = vector_store.search(query, k=k)
    anchors  = extract_entities(query)
    subgraph = kg.expand(anchors, hops=2)
    context  = [p.text for p in passages]
    context += [f"{e.render()}   (source: {e.evidence})" for e in subgraph]
    return rerank(query, dedupe(context))[:k]
```

**Graph RAG is strictly more expensive** — extraction is an LLM call per chunk, plus community summaries. Measure before adopting: build a question set with single-hop, multi-hop, and corpus-level questions, and compare vector-only / graph-only / hybrid on recall, faithfulness, and cost. If multi-hop is under ~10% of traffic, graph expansion *on demand* usually beats graph-everywhere.

⚠️ **The dangerous failure mode is a confidently wrong edge.** A missing chunk yields "I don't know"; a hallucinated `ACQUIRED` edge yields a fluent, cited, false answer. That is why every edge carries `evidence` and why extraction uses a closed relation vocabulary.

---
## Summary

| Concept | Why it matters |
|---|---|
| **State + reducers** | Parallel branches merge instead of clobbering |
| **Conditional edges** | Stop conditions become testable code, not prose |
| **Bounded cycles** | Retry/refine loops that provably terminate |
| **Checkpointers** | Crash resume, durable human interrupts, time-travel debugging |
| **Dynamic fan-out** | Map-reduce over an unknown number of items — with a *logged* cap |
| **Entity resolution** | The step that decides if the graph is queryable at all |
| **Evidence on edges** | Makes a wrong fact detectable instead of just fluent |
| **Local / global / path** | Three question shapes vector search handles badly |
| **Temporal edges** | Facts expire; overwriting them loses the audit trail |

**Loop or graph?** Loops (Module 13) when step count is unknown and the next action emerges from the last result. Graphs when the topology is known and you need auditability and replay. Production systems nest them.

---
## 🧪 Exercises

1. **Reducer bug**: remove the `docs` reducer in cell 2 and explain, in superstep terms, exactly where the first branch's docs are lost.
2. **Bounded cycle**: make `grade` always return `relevant=False`. Verify termination at `attempts == 3`, then delete the mechanical stop and observe what `max_steps` had to catch.
3. **Time travel**: replay from step 2 instead of step 1 and explain why the resumed frontier differs.
4. **Extraction schema**: extract from 5 paragraphs with an open-ended prompt vs. the closed enum. Count distinct relation types. Which graph can you query?
5. **Resolution cascade**: add "acme corp.", "Acme", "ACME" and confirm all collapse. Then construct a case where normalized matching merges two genuinely *different* companies — and decide what check would catch it.
6. **Multi-hop showdown**: write 15 questions (5 single-hop, 5 multi-hop, 5 corpus-level). Answer each with vector RAG, graph RAG, and hybrid. Report accuracy per category and cost — then decide what you'd ship.
7. **Failure injection**: insert one false edge, ask a question that traverses it, show the confident wrong answer, then show how `evidence` makes it detectable.